# Stage 2: Pairwise Alignment & Cross-Pillar Correlation

In this stage, we connect the pillars two-at-a-time to explore direct relationships:
1. **Rainfall vs. Food Prices**: Do climate shocks impact commodity prices?
2. **Food Prices vs. External Debt**: Does price volatility correlate with debt growth?
3. **Rainfall vs. External Debt**: Is there a direct link between climate and debt?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import grangercausalitytests
import os

sns.set(style="whitegrid")

## 2.1 Rainfall -> Food Price Alignment
Merging subnational rainfall with market-level food prices.

In [ ]:
# Load Data
rf_df = pd.read_csv('datasets/bgd-rainfall-subnat-full.csv')
food_df = pd.read_csv('datasets/wfp_food_prices_bgd.csv')

rf_df['date'] = pd.to_datetime(rf_df['date'])
food_df['date'] = pd.to_datetime(food_df['date'])

# Temporal and Spatial Alignment
rf_agg = rf_df.groupby(['date', 'admin1'])['rfq'].mean().reset_index()
food_agg = food_df.groupby(['date', 'admin1'])['price'].mean().reset_index()

merged_ab = pd.merge(rf_agg, food_agg, on=['date', 'admin1'])

plt.figure(figsize=(10, 6))
sns.regplot(data=merged_ab, x='rfq', y='price', scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title('Rainfall Anomaly vs Food Price: Direct Vulnerability Alignment')
plt.show()

## 2.2 Food Price -> External Debt Alignment
Analyzing annual food price volatility vs. debt growth rates.

In [ ]:
# Load Debt Data and Pivot
debt_df = pd.read_csv('datasets/external-debt_bgd.csv')
debt_filtered = debt_df[debt_df['Indicator Name'] == 'External debt stocks, total (DOD, current US$)'].copy()
debt_filtered = debt_filtered[['Year', 'Value']].rename(columns={'Year': 'year', 'Value': 'total_debt'})
debt_filtered['debt_growth'] = debt_filtered['total_debt'].pct_change()

# Annual Food Stats
food_df['year'] = food_df['date'].dt.year
food_annual = food_df.groupby('year')['price'].agg(['mean', 'std']).reset_index()
food_annual.columns = ['year', 'avg_food_price', 'food_volatility']

merged_bc = pd.merge(food_annual, debt_filtered, on='year').dropna()

# Cluster years by Food-Debt Risk
scaler = StandardScaler()
X_bc = scaler.fit_transform(merged_bc[['food_volatility', 'debt_growth']])
merged_bc['cluster'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_bc)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=merged_bc, x='food_volatility', y='debt_growth', hue='cluster', palette='viridis', s=100)
plt.title('Food Price Volatility vs External Debt Growth: Year Clusters')
plt.show()

## 2.3 Rainfall -> External Debt Alignment (Granger Causality)
Testing if annual rainfall patterns have a leading relationship with debt stock.

In [ ]:
# Annual Rainfall Stats
rf_df['year'] = rf_df['date'].dt.year
rf_annual = rf_df.groupby('year')['rfq'].mean().reset_index()

merged_ac = pd.merge(rf_annual, debt_filtered, on='year').dropna()

# Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(merged_ac.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Stage 2: Rainfall & Debt Indicators Correlation Heatmap')
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(merged_ac['year'], merged_ac['rfq'], label='Avg Rainfall Anomaly')
plt.twinx()
plt.plot(merged_ac['year'], merged_ac['total_debt'] / 1e9, color='orange', label='Total Debt (Billion $)')
plt.title('Annual Rainfall Trends vs External Debt Stock')
plt.show()

print("--- Granger Causality Test (Rain -> Debt) ---")
try:
    grangercausalitytests(merged_ac[['total_debt', 'rfq']], maxlag=2, verbose=True)
except Exception as e:
    print(f"Could not perform test: {e}")